In [1]:
import os
import pandas as pd
import re

base_path = "/Users/kirtisoglu/Documents/Documents/GitHub/FalcomChain/data/network-data/ttm_parquet"

travel_times = {}
origin_to_dest = {}

partition_re = re.compile(r"from_id=(\d+)")

for root, dirs, files in os.walk(base_path):
    match = partition_re.search(root)
    if not match:
        continue  # skip non-partition folders
    
    origin = int(match.group(1))
        
    for file in files:
        if file.endswith(".parquet"):
            df = pd.read_parquet(os.path.join(root, file), columns=["to_id", "travel_time_p50"])
            
            for dest, time in df.itertuples(index=False):
                travel_times[(origin, int(dest))] = int(time)
                
print(f"Loaded {len(travel_times)} OD pairs")


Loaded 49060046 OD pairs


In [2]:
origin_to_dest = {}

for pair in travel_times.keys():
    o = pair[0]
    d = pair[1]
    
    if o not in origin_to_dest.keys():
        origin_to_dest[o] = set()
    
    origin_to_dest[o].add(d)

In [3]:
from falcomchain.helper import DataHandler

handler = DataHandler()
graph = handler.load_graphhh()

missing = {}
nodes = set(graph.nodes)


for origin, destinations in origin_to_dest.items():
    missing[origin] =  nodes - destinations

for key, value in missing.items():
    if value != {30185, 32588}:
        print(key)
        print(value)

30185
{51, 52, 53, 54, 55, 56, 59, 72, 73, 74, 75, 76, 77, 78, 79, 80, 81, 82, 83, 84, 85, 86, 87, 88, 89, 90, 91, 92, 93, 94, 95, 96, 97, 98, 99, 100, 101, 102, 103, 104, 105, 106, 107, 108, 109, 110, 111, 112, 113, 114, 115, 116, 117, 118, 119, 120, 121, 122, 127, 159, 160, 161, 162, 175, 176, 177, 178, 179, 182, 183, 185, 187, 188, 189, 196, 197, 198, 200, 201, 202, 203, 204, 205, 258, 259, 272, 273, 274, 275, 291, 292, 293, 294, 295, 296, 297, 298, 299, 300, 301, 302, 409, 410, 411, 412, 416, 417, 418, 424, 425, 426, 427, 428, 434, 469, 470, 503, 504, 505, 506, 507, 508, 509, 510, 511, 512, 513, 525, 526, 527, 528, 529, 530, 533, 534, 535, 536, 537, 538, 539, 540, 541, 542, 543, 544, 545, 546, 547, 548, 549, 550, 551, 553, 556, 565, 566, 567, 568, 574, 575, 585, 586, 587, 588, 589, 591, 593, 598, 599, 601, 611, 623, 637, 638, 639, 640, 641, 642, 643, 644, 645, 646, 647, 648, 649, 650, 651, 652, 653, 654, 655, 656, 657, 658, 659, 660, 661, 662, 663, 664, 665, 666, 667, 668, 669, 670

In [5]:
disconnected = {30185, 32588}

# change the graph candidate attributes of the disconnected origins
for node in disconnected:
    graph.nodes[node]['candidate'] = 0
    graph.nodes[node]['artificial_candidate'] = 0

# remove disconnected origins from travel_times
remove = set()
for pair in travel_times.keys():
    o = pair[0]
    d = pair[1]
    
    if o in disconnected:
        remove.add(pair)
    
    elif d in disconnected:
        travel_times[pair] = 0

for pair in remove:
    del travel_times[pair]

In [13]:
origin_df = pd.read_csv("/Users/kirtisoglu/Documents/Documents/GitHub/FalcomChain/data/network-data/origins.csv")
origins = set(origin_df['id'])
dests_df = pd.read_csv("/Users/kirtisoglu/Documents/Documents/GitHub/FalcomChain/data/network-data/destinations.csv")
dests = set(dests_df['id'])


expected = len(graph.nodes)*(len(origin_df)-2)
print("expected number of pairs", expected)
print("difference", expected - len(travel_times))



expected number of pairs 49065540
difference 5496


In [12]:
import pickle
path = "/Users/kirtisoglu/Documents/Documents/GitHub/FalcomChain/data/processed/real_travel_times.pkl"
with open(path, "wb") as file:
    pickle.dump(travel_times, file)

In [7]:
for origin, dests in missing.items():
    
    #if len(missings[origin]) > 0:
        #print(missings[origin])
    
    if dests != {30185, 32588}:
        print(f"{origin} has the missing set {dests}")

30185 has the missing set {51, 52, 53, 54, 55, 56, 59, 72, 73, 74, 75, 76, 77, 78, 79, 80, 81, 82, 83, 84, 85, 86, 87, 88, 89, 90, 91, 92, 93, 94, 95, 96, 97, 98, 99, 100, 101, 102, 103, 104, 105, 106, 107, 108, 109, 110, 111, 112, 113, 114, 115, 116, 117, 118, 119, 120, 121, 122, 127, 159, 160, 161, 162, 175, 176, 177, 178, 179, 182, 183, 185, 187, 188, 189, 196, 197, 198, 200, 201, 202, 203, 204, 205, 258, 259, 272, 273, 274, 275, 291, 292, 293, 294, 295, 296, 297, 298, 299, 300, 301, 302, 409, 410, 411, 412, 416, 417, 418, 424, 425, 426, 427, 428, 434, 469, 470, 503, 504, 505, 506, 507, 508, 509, 510, 511, 512, 513, 525, 526, 527, 528, 529, 530, 533, 534, 535, 536, 537, 538, 539, 540, 541, 542, 543, 544, 545, 546, 547, 548, 549, 550, 551, 553, 556, 565, 566, 567, 568, 574, 575, 585, 586, 587, 588, 589, 591, 593, 598, 599, 601, 611, 623, 637, 638, 639, 640, 641, 642, 643, 644, 645, 646, 647, 648, 649, 650, 651, 652, 653, 654, 655, 656, 657, 658, 659, 660, 661, 662, 663, 664, 665, 666

In [8]:
import pandas as pd
origin_df = pd.read_csv("/Users/kirtisoglu/Documents/Documents/GitHub/FalcomChain/data/network-data/origins.csv")
origins = set(origin_df['id'])
dests_df = pd.read_csv("/Users/kirtisoglu/Documents/Documents/GitHub/FalcomChain/data/network-data/destinations.csv")
dests = set(dests_df['id'])

nodes = {8898, 1624}
for node in nodes:
    
    if node in origins:
        print(node, "in origins")
    if node in dests:
        print(node, "in destinations")

1624 in origins
1624 in destinations
8898 in destinations


In [31]:
nodes_2 = {17222, 25049}
nodes_3 = {9159, 3625}

for node in nodes_3:
    
    if node in origins:
        print(node, "in origins")
    if node in dests:
        print(node, "in destinations")

3625 in origins
3625 in destinations
9159 in destinations


In [15]:
disconnected = {30185, 32588}
origins = set(origin_df['id'])

for node in disconnected:
    origins.remove(node)

In [21]:
for pair in travel_times.keys():
    
    o = pair[0]
    d = pair[1]
    
    if d in disconnected:
        travel_times[pair] = 0

In [26]:
candidates = {node for node in graph.nodes if graph.nodes[node]['candidate'] == 1}
print(len(origins))
print(len(candidates))

2748
2750


In [ ]:
nodes_3 = {9159, 3625}

In [29]:
final_missing = set()
final_origin_to_dests = {}

for u in candidates:
    if u not in final_origin_to_dests.keys():
        final_origin_to_dests[u] = set()
        
    for v in graph.nodes:
        pair = (u,v)
        if pair not in travel_times.keys():
            if v == 30185 or 32588:
                travel_times[pair] = 0
            else:
                final_missing.add((u,v))
                final_origin_to_dests[u].add(v)

print(len(final_missing))

0


In [30]:
import pickle
path = "/Users/kirtisoglu/Documents/Documents/GitHub/FalcomChain/data/processed/real_travel_times.pkl"
with open(path, "wb") as file:
    pickle.dump(travel_times, file)

In [11]:
from falcomchain.helper import DataHandler

handler = DataHandler()
graph = handler.load_graphhh()


for node in nodes:
    print(node)
    print(graph.nodes[node])
    

print(travel_times[(1624, 8898)])

1624
{'boundary_node': False, 'area': 7721.692618246345, 'GEOID20': '170316603021005', 'population': 80, 'C_X': -87.69447655540547, 'C_Y': 41.785254987544285, 'candidate': 1, 'real_candidate': 0, 'artificial_candidate': 1}
8898
{'boundary_node': False, 'area': 7638.969636994795, 'GEOID20': '170318350003002', 'population': 93, 'C_X': -87.6976853132744, 'C_Y': 41.79089047785875, 'candidate': 0, 'real_candidate': 0, 'artificial_candidate': 0}
4
